# Attention Core: QKV, Prefill, And Decode

This notebook builds the core attention path used by decoder-only LLMs.

We implement vectorized attention, run causal prefill, run token-by-token decode
with an explicit KV cache, and verify that both paths produce the same result.

No `attention_forge` imports.

## Attention variants:

**MHA: Multi-Head Attention.** Runs several attention heads in parallel so different heads can learn different query-key-value projections. Standard decoder attention stores one K/V head per query head.

**MQA: Multi-Query Attention.** Keeps many query heads but shares a single K/V head across them, reducing KV-cache memory and decode reads.

**GQA: Grouped-Query Attention.** Middle ground between MHA and MQA: groups of query heads share fewer K/V heads.

**MLA: Multi-Head Latent Attention.** Stores compressed latent cache state instead of full per-head K/V tensors.

**DSA/CSA: DeepSeek Sparse Attention / Compressed Sparse Attention.** Selects a subset of prior tokens or compressed blocks so decode attention reads less context.


## Setup

In [1]:
import math

import torch
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)


def print_table(headers, rows):
    widths = [len(str(header)) for header in headers]
    for row in rows:
        for idx, value in enumerate(row):
            widths[idx] = max(widths[idx], len(str(value)))
    template = "  ".join(f"{{:<{width}}}" for width in widths)
    print(template.format(*headers))
    print(template.format(*["-" * width for width in widths]))
    for row in rows:
        print(template.format(*[str(value) for value in row]))

## Shape Contract

Notation used in the code below:

- `B`: batch size
- `T`: sequence length, or number of tokens
- `H`: number of attention heads
- `D`: dimension per head
- `C`: hidden size / model width, so `C = H * D`

We use the common attention layout after projection:

```text
q, k, v: [batch, heads, tokens, head_dim]
```

For MHA, query heads and KV heads are the same count. MQA/GQA relax this later.

In [2]:
# B: batch size. We keep it >1 so tensor code is truly batched.
B = 2
# T: sequence length, meaning how many token positions are processed.
T = 16
# H: number of attention heads.
H = 4
# D: per-head dimension.
D = 8
# C: hidden size / model width. Splitting into heads gives H * D.
C = H * D

shape_rows = [
    ["hidden states x", [B, T, C]],
    ["q after split", [B, H, T, D]],
    ["k after split", [B, H, T, D]],
    ["v after split", [B, H, T, D]],
    ["attention scores", [B, H, T, T]],
    ["attention output", [B, H, T, D]],
]
print_table(["tensor", "shape"], shape_rows)

tensor            shape         
----------------  --------------
hidden states x   [2, 16, 32]   
q after split     [2, 4, 16, 8] 
k after split     [2, 4, 16, 8] 
v after split     [2, 4, 16, 8] 
attention scores  [2, 4, 16, 16]
attention output  [2, 4, 16, 8] 


## Vectorized Scaled Dot-Product Attention

The mask is applied to logits before softmax. Masked logits become `-inf`; after
softmax, those positions have probability zero.

In [3]:
def causal_mask(q_len, k_len, device):
    return torch.ones(q_len, k_len, dtype=torch.bool, device=device).tril()


def manual_attention(q, k, v, *, causal):
    head_dim = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(head_dim)
    if causal:
        mask = causal_mask(q.shape[-2], k.shape[-2], q.device)
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    out = weights @ v
    return out, weights

## Split And Merge Heads

These two functions are easy to get wrong. Most implementation bugs in toy
attention come from layout mistakes here.

In [4]:
def split_heads(x, num_heads):
    batch, tokens, hidden = x.shape
    assert hidden % num_heads == 0
    head_dim = hidden // num_heads
    x = x.view(batch, tokens, num_heads, head_dim)
    return x.transpose(1, 2).contiguous()


def merge_heads(x):
    batch, heads, tokens, head_dim = x.shape
    x = x.transpose(1, 2).contiguous()
    return x.view(batch, tokens, heads * head_dim)


x = torch.randn(B, T, C)
x_roundtrip = merge_heads(split_heads(x, H))
print(torch.allclose(x, x_roundtrip))

True


## MHA Prefill

Prefill processes the full prompt in one call and uses a causal mask.

In [5]:
def project_qkv(x, wq, wk, wv, num_heads):
    # x is [B, T, C]. Projection keeps C, then split_heads exposes [B, H, T, D].
    q = split_heads(x @ wq, num_heads)
    k = split_heads(x @ wk, num_heads)
    v = split_heads(x @ wv, num_heads)
    return q, k, v


def mha_prefill(x, wq, wk, wv, wo, num_heads):
    # Full prompt path: all T tokens are present, so causal masking is required.
    q, k, v = project_qkv(x, wq, wk, wv, num_heads)
    attn_out, weights = manual_attention(q, k, v, causal=True)
    y = merge_heads(attn_out) @ wo
    return y, weights, k, v


scale = 1 / math.sqrt(C)
wq = torch.randn(C, C) * scale
wk = torch.randn(C, C) * scale
wv = torch.randn(C, C) * scale
wo = torch.randn(C, C) * scale
x = torch.randn(B, T, C)

prefill_y, prefill_weights, prefill_k, prefill_v = mha_prefill(x, wq, wk, wv, wo, H)
print("prefill_y", list(prefill_y.shape))
print("prefill_k", list(prefill_k.shape))
print("prefill_v", list(prefill_v.shape))

prefill_y [2, 16, 32]
prefill_k [2, 4, 16, 8]
prefill_v [2, 4, 16, 8]


## MHA Decode With Explicit KV Cache

Decode processes one token. The cache contains only previous tokens plus the
current token after append, so the single-token query does not need a future mask.

In [7]:
def append_cache(cache, new_value):
    # KV cache grows along token dimension: [B, H, cache_len, D].
    if cache is None:
        return new_value
    return torch.cat([cache, new_value], dim=2)


def mha_decode_step(x_t, wq, wk, wv, wo, num_heads, cache_k=None, cache_v=None):
    # x_t is one token: [B, 1, C]. Its query reads every cached key/value.
    q_t, k_t, v_t = project_qkv(x_t, wq, wk, wv, num_heads)
    cache_k = append_cache(cache_k, k_t)
    cache_v = append_cache(cache_v, v_t)
    attn_out, weights = manual_attention(q_t, cache_k, cache_v, causal=False)
    y_t = merge_heads(attn_out) @ wo
    return y_t, weights, cache_k, cache_v


cache_k = None
cache_v = None
decode_outputs = []
cache_lengths = []
for t in range(T):
    x_t = x[:, t : t + 1, :]
    y_t, weights_t, cache_k, cache_v = mha_decode_step(
        x_t,
        wq,
        wk,
        wv,
        wo,
        H,
        cache_k,
        cache_v,
    )
    decode_outputs.append(y_t)
    cache_lengths.append(cache_k.shape[2])

decode_y = torch.cat(decode_outputs, dim=1)
print("decode_y", list(decode_y.shape))
print("cache lengths", cache_lengths[:5], "...", cache_lengths[-1])
print("final cache", list(cache_k.shape), list(cache_v.shape))

decode_y [2, 16, 32]
cache lengths [1, 2, 3, 4, 5] ... 16
final cache [2, 4, 16, 8] [2, 4, 16, 8]


## Prefill/Decode Equivalence Check

With the same weights, no dropout, and no position encoding complications, full
causal prefill and token-by-token decode should produce the same outputs.

In [8]:
max_abs_error = (prefill_y - decode_y).abs().max().item()
print(f"max_abs_error = {max_abs_error:.8f}")
print("allclose", torch.allclose(prefill_y, decode_y, atol=1e-5, rtol=1e-5))

max_abs_error = 0.00000042
allclose True


## Cross-Check With PyTorch SDPA

Manual attention should match PyTorch's optimized API for this small case. This
is a useful guardrail before we move the implementation into package code.

In [ ]:
q, k, v = project_qkv(x, wq, wk, wv, H)
manual_out, _ = manual_attention(q, k, v, causal=True)
torch_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("manual vs torch SDPA", torch.allclose(manual_out, torch_out, atol=1e-5, rtol=1e-5))
print("max_abs_error", (manual_out - torch_out).abs().max().item())

## What Matters For The Next Notebook

The cache is not an abstract idea. In decode we explicitly accumulated:

```text
cache_k: [batch, heads, cache_length, head_dim]
cache_v: [batch, heads, cache_length, head_dim]
```

Every generated token appends one K and one V per layer. Every following token
reads that larger cache. The next notebook turns this into memory and bandwidth
math for MHA, MQA, GQA, MLA, and sparse attention.